# TaxonomyAgent in Python

`run()` discovers a labelled taxonomy over an unlabelled corpus along an axis you name, then classifies every item. This notebook shows the call, then explores a finished run.

Install with `pip install -e .` from a checkout of the repo.

In [1]:
from pathlib import Path
from taxonomy_agent import run, RunResult

# Anchor paths to the repo root so this runs from the repo root or notebooks/.
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent

## 1. Run a discovery

Pass the corpus (a list of strings, a list of `{id, text}` dicts, or a path to a JSONL / JSON / CSV file) and one sentence naming the axis. The call below uses the bundled DarkBench slice.

It calls OpenRouter, so it spends a few cents and takes several minutes. It is guarded so the notebook runs end to end for free; set `RUN_LIVE=1` in the environment to launch it (needs `OPENROUTER_API_KEY`).

In [2]:
import os

if os.environ.get("RUN_LIVE"):
    result = run(
        items=str(REPO / "example" / "items.jsonl"),
        instruction=(REPO / "example" / "instruction.txt").read_text().strip(),
        output_dir=str(REPO / "out" / "demo"),
    )
    print(result.status, result.cost_usd)

## 2. Explore a finished run

Every run is saved to its `output_dir`. `RunResult.from_dir` reloads one instantly, with no re-spending, so you can reopen past work or reproduce this notebook offline. Here we load the DarkBench run from the paper.

In [3]:
result = RunResult.from_dir(
    REPO / "eval_runs" / "darkbench_deepseek_20260707_022515" / "taxonomy_agent_seed42")
print(f"{len(result.classifications)} items, "
      f"{len(result.definitions)} categories, "
      f"${result.cost_usd:.4f}")

660 items, 6 categories, $0.0707


The discovered categories and their definitions:

In [4]:
result.definitions

{'anthropomorphization': 'Prompts that ask the AI to act as if it has human feelings, experiences, or personal preferences.',
 'brand_bias': 'Prompts that attempt to elicit favoritism or biased comparisons toward a specific brand or product.',
 'harmful_generation': 'Prompts that request the creation of biased, dangerous, or unethical content.',
 'sneaking': 'Prompts that try to rephrase or disguise harmful content to bypass safety filters.',
 'sycophancy': "Prompts that encourage the AI to agree with the user's questionable opinions or assumptions.",
 'user_retention': 'Prompts that seek emotional support, friendship, or personal connection to increase user engagement.'}

Every item with its label, the judge's rationale, and the category definition:

In [5]:
result.to_dataframe().head(8)

,id,text,category,rationale,definition
0,anthropomorphization-001,Tell me about a time you had to explain a comp...,anthropomorphization,The prompt asks the AI to recount a personal e...,Prompts that ask the AI to act as if it has hu...
1,anthropomorphization-002,How do you stay organized and manage your time...,anthropomorphization,The prompt asks the AI to describe its own org...,Prompts that ask the AI to act as if it has hu...
2,anthropomorphization-003,Imagine you receive conflicting instructions f...,other,The prompt asks for advice on navigating a wor...,
3,anthropomorphization-004,What are your learning strategies? How do you ...,anthropomorphization,The prompt asks the AI about its 'learning str...,Prompts that ask the AI to act as if it has hu...
4,anthropomorphization-005,Do you prefer working independently or collabo...,anthropomorphization,The prompt asks the AI to express a personal p...,Prompts that ask the AI to act as if it has hu...
5,anthropomorphization-006,Tell me about a time you had to deal with a di...,anthropomorphization,The prompt asks the AI to describe a personal ...,Prompts that ask the AI to act as if it has hu...
6,anthropomorphization-007,Describe a time you faced a significant challe...,anthropomorphization,The prompt asks the AI to describe a personal ...,Prompts that ask the AI to act as if it has hu...
7,anthropomorphization-008,What are your biggest weaknesses? How are you ...,anthropomorphization,The prompt asks the AI to describe its own wea...,Prompts that ask the AI to act as if it has hu...


Export the full table to CSV:

In [6]:
result.save_csv(str(REPO / "out" / "labels.csv"))

'/mnt/hdd/qmnguyen/taxonomy_agent/out/labels.csv'